
# AI for Loan Approval (loan_data.csv)
## Compact pipeline: compare models, choose best, then run Gradio app

This notebook does 2 things:
1. Train and compare several models (machine learning + a small neural network).
2. Use the best model in a Gradio loan application form.

What we predict:
- `loan_status = 1` means Approved
- `loan_status = 0` means Not Approved




---
## Step 1: Imports


In [1]:
#Importing necessary libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

# Metrics are measurable indicators that evaluate a model’s performance, and they are important because they help determine how accurately, fairly, and effectively the model makes decisions.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)



---
## Step 2: Load `loan_data.csv`
In Colab, upload `loan_data.csv` when prompted.


In [2]:

from google.colab import files
uploaded = files.upload()

csv_name = list(uploaded.keys())[0]
df = pd.read_csv(csv_name)

print("Rows, columns:", df.shape)
df.head()


Saving loan_data.csv to loan_data.csv
Rows, columns: (45000, 14)


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1



---
## Step 3: Choose realistic loan application inputs + add one backend feature

We use common application fields (things a loan officer would ask):
- annual income
- employment experience (years)
- home ownership
- requested loan amount
- loan intent (purpose)
- credit score and credit history length
- whether there were previous defaults on file

We also calculate one helpful ratio in the backend:
- `loan_percent_income = loan_amnt / person_income`



In [3]:

TARGET = "loan_status"

USER_INPUT_COLS = [
    "person_income",
    "person_emp_exp",
    "person_home_ownership",
    "loan_amnt",
    "loan_intent",
    "cb_person_cred_hist_length",
    "credit_score",
    "previous_loan_defaults_on_file",
]

missing = [c for c in USER_INPUT_COLS + [TARGET] if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

def add_backend_features(X):
    X = X.copy()
    eps = 1e-9
    X["loan_percent_income"] = X["loan_amnt"] / (X["person_income"] + eps)
    return X

y = df[TARGET].astype(int)
X = add_backend_features(df[USER_INPUT_COLS].copy())

print("Class balance (0/1):")
print(y.value_counts())
X.head()


Class balance (0/1):
loan_status
0    35000
1    10000
Name: count, dtype: int64


,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_percent_income
0,71948.0,0,RENT,35000.0,PERSONAL,3.0,561,No,0.486462
1,12282.0,0,OWN,1000.0,EDUCATION,2.0,504,Yes,0.081420
2,12438.0,3,MORTGAGE,5500.0,MEDICAL,3.0,635,No,0.442193
3,79753.0,0,RENT,35000.0,MEDICAL,2.0,675,No,0.438855
4,66135.0,1,RENT,35000.0,MEDICAL,4.0,586,No,0.529221



---
## Step 4: Train / Validation split
We use the validation set to choose the best model.


In [4]:

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
)
print("Train:", X_train.shape, "Val:", X_val.shape)


Train: (33750, 9) Val: (11250, 9)



---
## Step 5: Preprocessing + scoring

Preprocessing:
- Numeric columns: fill missing values (median), then scale
- Categorical columns: fill missing values (most common), then one-hot encode

Model selection metric:
- AUC (higher is better)


In [5]:

# OneHotEncoder option differs across scikit-learn versions
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", ohe)]), cat_cols),
])

def score_row(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "AUC": roc_auc_score(y_true, prob),
    }



---
## Step 6: Try multiple machine learning models
We train several models and compare validation AUC.


In [6]:

ml_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=RANDOM_SEED),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=12, random_state=RANDOM_SEED, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, max_depth=12, random_state=RANDOM_SEED, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_SEED),
}

ml_rows, ml_pipes = [], {}

for name, est in ml_models.items():
    pipe = Pipeline([("prep", preprocess), ("model", clone(est))]).fit(X_train, y_train)
    prob = pipe.predict_proba(X_val)[:, 1]
    row = score_row(y_val, prob, threshold=0.5)
    row["Model"] = name
    ml_rows.append(row)
    ml_pipes[name] = pipe

results_ml = pd.DataFrame(ml_rows).sort_values("AUC", ascending=False)
results_ml


,Accuracy,Precision,Recall,F1,AUC,Model
4,0.904000,0.851137,0.6884,0.761168,0.960749,HistGradientBoosting
2,0.895911,0.878201,0.6172,0.724924,0.953446,Random Forest
3,0.878756,0.784000,0.6272,0.696889,0.941040,Extra Trees
1,0.887822,0.819401,0.6352,0.715638,0.938786,Decision Tree
0,0.869867,0.735883,0.6464,0.688245,0.935316,Logistic Regression



---
## Step 7: Try deep learning (small neural network)
We train a small neural network and compare validation AUC.


In [7]:

import tensorflow as tf
tf.random.set_seed(RANDOM_SEED)

prep_nn = clone(preprocess).fit(X_train)
Xtr = prep_nn.transform(X_train).astype("float32")
Xva = prep_nn.transform(X_val).astype("float32")

nn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(Xtr.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
nn.compile(optimizer="adam", loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name="auc")])

nn.fit(Xtr, y_train, validation_data=(Xva, y_val), epochs=10, batch_size=256, verbose=0)

prob_nn = nn.predict(Xva).ravel()
nn_row = score_row(y_val, prob_nn, threshold=0.5)
nn_row["Model"] = "Neural Network"
nn_row


352/352 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


{'Accuracy': 0.8836444444444445,
 'Precision': 0.800302571860817,
 'Recall': 0.6348,
 'F1': 0.708008030336828,
 'AUC': np.float64(0.9439896457142858),
 'Model': 'Neural Network'}


---
## Step 8: Choose the best model (highest validation AUC)


In [8]:

all_results = pd.concat([results_ml, pd.DataFrame([nn_row])], ignore_index=True).sort_values("AUC", ascending=False)
all_results


,Accuracy,Precision,Recall,F1,AUC,Model
0,0.904000,0.851137,0.6884,0.761168,0.960749,HistGradientBoosting
1,0.895911,0.878201,0.6172,0.724924,0.953446,Random Forest
5,0.883644,0.800303,0.6348,0.708008,0.943990,Neural Network
2,0.878756,0.784000,0.6272,0.696889,0.941040,Extra Trees
3,0.887822,0.819401,0.6352,0.715638,0.938786,Decision Tree
4,0.869867,0.735883,0.6464,0.688245,0.935316,Logistic Regression


In [9]:

best_name = all_results.iloc[0]["Model"]
print("Best model:", best_name)


Best model: HistGradientBoosting



---
## Gradio app (required fields, decision only)

This app uses the best model chosen above.
It asks common questions and outputs only:
- Approved
- Not Approved

All fields are required:
- If any field is missing, the app shows an error message.


In [10]:

!pip -q install gradio
import gradio as gr


In [11]:
import gradio as gr
import pandas as pd

# Use the best model in memory
USE_THRESHOLD = 0.1
MIN_CREDIT_SCORE = 600  # simple guardrail
MIN_SALARY = 30000 # new guardrail: minimum annual income

# Ensure 'df' is defined. If the kernel was restarted, 'df' might be lost.
# This part is added to resolve 'NameError: name 'df' is not defined'.
if 'df' not in locals() and 'df' not in globals():
    try:
        # Based on previous execution output, the file was saved as 'loan_data (1).csv'.
        df = pd.read_csv('loan_data (1).csv')
        print("DataFrame 'df' reloaded from 'loan_data (1).csv'.")
    except FileNotFoundError:
        try:
            # Fallback for original name if the renamed one is not found.
            df = pd.read_csv('loan_data.csv')
            print("DataFrame 'df' reloaded from 'loan_data.csv'.")
        except FileNotFoundError:
            # If neither file is found, it means the data wasn't successfully uploaded/saved or kernel state is severely compromised.
            # Raising a Gradio error to inform the user.
            raise gr.Error("Error: 'loan_data.csv' or 'loan_data (1).csv' not found. Please ensure the CSV file is uploaded and available by re-running previous cells.")

home_choices = sorted(df["person_home_ownership"].astype(str).unique().tolist())
intent_choices = sorted(df["loan_intent"].astype(str).unique().tolist())
default_choices = sorted(df["previous_loan_defaults_on_file"].astype(str).unique().tolist())

def require(value, name):
    # Treat None or empty string as missing
    if value is None:
        raise gr.Error(f"Please enter: {name}")
    if isinstance(value, str) and value.strip() == "":
        raise gr.Error(f"Please enter: {name}")
    return value

def predict_decision(person_income, person_emp_exp, person_home_ownership,
                     loan_amnt, loan_intent, cb_person_cred_hist_length,
                     credit_score, previous_loan_defaults_on_file):

    # Required-field checks
    person_income = float(require(person_income, "Annual income"))
    person_emp_exp = float(require(person_emp_exp, "Employment experience (years)"))
    person_home_ownership = str(require(person_home_ownership, "Home ownership"))
    loan_amnt = float(require(loan_amnt, "Requested loan amount"))
    loan_intent = str(require(loan_intent, "Loan intent"))
    cb_person_cred_hist_length = float(require(cb_person_cred_hist_length, "Credit history length (years)"))
    credit_score = float(require(credit_score, "Credit score"))
    previous_loan_defaults_on_file = str(require(previous_loan_defaults_on_file, "Previous defaults on file"))

    # Build 1-row DataFrame
    X1 = pd.DataFrame([
        {
            "person_income": person_income,
            "person_emp_exp": person_emp_exp,
            "person_home_ownership": person_home_ownership,
            "loan_amnt": loan_amnt,
            "loan_intent": loan_intent,
            "cb_person_cred_hist_length": cb_person_cred_hist_length,
            "credit_score": credit_score,
            "previous_loan_defaults_on_file": previous_loan_defaults_on_file,
        }
    ])

    # Backend feature
    # Note: add_backend_features must be defined in the global scope for this to work correctly.
    X1 = add_backend_features(X1)

    # Predict probability using the chosen best model
    # Note: best_name, nn, prep_nn, ml_pipes must be defined in the global scope for this to work correctly.
    if best_name == "Neural Network":
        prob = float(nn.predict(prep_nn.transform(X1).astype("float32")).ravel()[0])
    else:
        prob = float(ml_pipes[best_name].predict_proba(X1)[:, 1][0])

    # Convert probability to decision
    approved = (prob >= USE_THRESHOLD)

    # Simple guardrails for realism (override approval)
    if credit_score < MIN_CREDIT_SCORE:
        approved = False

    # New guardrail: minimum annual income
    if person_income < MIN_SALARY:
        approved = False

    # Also override if defaults are "Yes" (handle different text formats)
    if previous_loan_defaults_on_file.strip().lower() in ["yes", "1", "true"]:
        approved = False

    return "Approved" if approved else "Not Approved"

demo = gr.Interface(
    fn=predict_decision,
    inputs=[
        gr.Number(label="Annual income"),
        gr.Number(label="Employment experience (years)"),
        gr.Dropdown(home_choices, value=None, label="Home ownership"),
        gr.Number(label="Requested loan amount"),
        gr.Dropdown(intent_choices, value=None, label="Loan intent"),
        gr.Number(label="Credit history length (years)"),
        gr.Number(label="Credit score"),
        gr.Dropdown(default_choices, value=None, label="Previous defaults on file"),
    ],
    outputs=gr.Textbox(label="Decision"),
    title="Loan Approval (Simple Demo)",
    description="All fields are required. Output is only Approved or Not Approved."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://674f66ce2c7d4d1bb2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
